In [6]:
import sparknlp
from sparknlp.base import *
from sparknlp.annotator import *

spark = sparknlp.start(gpu=True)

ModuleNotFoundError: No module named 'distutils'

In [2]:
df = spark.read.option("header","true").csv("../Greek_Parliament_Proceedings_1989_2020/Greek_Parliament_Proceedings_1989_2020.csv")
dropped_columns = ["parliamentary_period","parliamentary_session","parliamentary_sitting","government","member_region","roles"]
wanted_columns_df = df.drop(*dropped_columns)

In [3]:
wanted_columns_df.show(5)

+--------------------+------------+--------------------+-------------+--------------------+
|         member_name|sitting_date|     political_party|member_gender|              speech|
+--------------------+------------+--------------------+-------------+--------------------+
|κρητικος νικολαου...|  03/07/1989|πανελληνιο σοσιαλ...|         male| Παρακαλείται ο Γ...|
|κρητικος νικολαου...|  03/07/1989|πανελληνιο σοσιαλ...|         male|" Παρακαλείται ο ...|
|κρητικος νικολαου...|  03/07/1989|πανελληνιο σοσιαλ...|         male| Κύριοι συνάδελφο...|
|                NULL|  03/07/1989|               βουλη|         NULL|   Μάλιστα, μάλιστα.|
|κρητικος νικολαου...|  03/07/1989|πανελληνιο σοσιαλ...|         male| Η Βουλή παρέσχε ...|
+--------------------+------------+--------------------+-------------+--------------------+
only showing top 5 rows



In [4]:
wanted_columns_df.select("speech").take(5)

[Row(speech=' Παρακαλείται ο Γραμματέας κ. Βουλγαράκης να συνοδεύσει το Μακαριότατο Αρχιεπίσκοπο Αθηνών και πάσης Ελλάδος κ. ΣΕΡΑΦΕΙΜ και τα συνοδεύοντα αυτόν μέλη της Ιεράς Συνόδου κατά την είσοδό τους στην Αίθουσα της Βουλής, προκειμένου να τελεσθεί αγιασμός.  . Στη συνέχεια τελείται ο καθιερωμένος αγιασμός.'),
 Row(speech='" Παρακαλείται ο κύριος Γραμματέας να συνοδεύσει την Ιερά Σύνοδο εκτός της Αιθούσης της Βουλής.  . Παρακαλείται ο συνάδελφος Βουλευτής κ. Σαδίκ Αμέτ, που ανήκει στο Μωαμεθανικό Θρήσκευμα να προσέλθει και να δώσει τον οριζόμενο από το Σύνταγμα όρκο επί του Κορανίου.  : ~""Ορκίζομαι στο όνομα του Παντοδύναμου Θεού και του μόνου αυτού Προφήτη ο οποίος είναι ο Μωάμεθ να είμαι πιστός στην πατρίδα και το δημοκρατικό πολίτευμα'),
 Row(speech=' Κύριοι συνάδελφοι, παρακαλώ τη Βουλή να εξουσιοδοτήσει το Προεδρείο για την επικύρωση των Πρακτικών της σημερινής συνεδριάσεως.'),
 Row(speech=' Μάλιστα, μάλιστα.'),
 Row(speech=' Η Βουλή παρέσχε τη ζητηθείσα εξουσιοδότηση. Με τη σ

In [5]:
document_assembler = DocumentAssembler() \
    .setInputCol("speech") \
	.setOutputCol("document")
tokenizer = Tokenizer() \
    .setInputCols(["document"]) \
	.setOutputCol("token")
normalizer = Normalizer() \
    .setInputCols(["token"]) \
    .setOutputCol("normalized") \
    .setLowercase(True)
stopwords = StopWordsCleaner().pretrained("stopwords_iso", "el") \
    .setInputCols(["normalized"]) \
    .setOutputCol("clean_normalized") \
    .setCaseSensitive(False)
lemmatizer = LemmatizerModel.pretrained("lemma", "el") \
    .setInputCols(["clean_normalized"]) \
    .setOutputCol("lemma")
finisher = Finisher() \
    .setInputCols(["lemma"]) \
    .setOutputCols(["tokens"]) \
	.setCleanAnnotations(False) \
	.setOutputAsArray(False) \
	.setAnnotationSplitSymbol(" ")

pipeline = Pipeline(stages=[
    document_assembler,
    tokenizer,
    normalizer,
    stopwords,
    lemmatizer,
    finisher
])

stopwords_iso download started this may take some time.
Approximate size to download 3.7 KB
[ | ]

26/01/18 18:02:05 WARN S3AbortableInputStream: Not all bytes were read from the S3ObjectInputStream, aborting HTTP connection. This is likely an error and may result in sub-optimal behavior. Request only the bytes you need via a ranged GET or drain the input stream after use.
26/01/18 18:02:06 WARN S3AbortableInputStream: Not all bytes were read from the S3ObjectInputStream, aborting HTTP connection. This is likely an error and may result in sub-optimal behavior. Request only the bytes you need via a ranged GET or drain the input stream after use.


stopwords_iso download started this may take some time.
Approximate size to download 3.7 KB
Download done! Loading the resource.
[OK!]
lemma download started this may take some time.
Approximate size to download 123.8 KB
[ | ]

26/01/18 18:02:08 WARN S3AbortableInputStream: Not all bytes were read from the S3ObjectInputStream, aborting HTTP connection. This is likely an error and may result in sub-optimal behavior. Request only the bytes you need via a ranged GET or drain the input stream after use.
26/01/18 18:02:08 WARN S3AbortableInputStream: Not all bytes were read from the S3ObjectInputStream, aborting HTTP connection. This is likely an error and may result in sub-optimal behavior. Request only the bytes you need via a ranged GET or drain the input stream after use.
26/01/18 18:02:09 WARN S3AbortableInputStream: Not all bytes were read from the S3ObjectInputStream, aborting HTTP connection. This is likely an error and may result in sub-optimal behavior. Request only the bytes you need via a ranged GET or drain the input stream after use.


lemma download started this may take some time.
Approximate size to download 123.8 KB
Download done! Loading the resource.
[OK!]


In [6]:
model = pipeline.fit(wanted_columns_df)
processed_df = model.transform(wanted_columns_df)
clean_df = processed_df.dropna().drop("speech")

In [8]:
from pyspark.ml.feature import Tokenizer, CountVectorizer, IDF
from pyspark.ml import Pipeline

ml_tokenizer = Tokenizer(inputCol="tokens", outputCol="words")

vectorizer = CountVectorizer(inputCol="words", outputCol="rawFeatures", vocabSize=100_000)
idf = IDF(inputCol="rawFeatures", outputCol="features")
ml_pipeline = Pipeline(stages=[ml_tokenizer, vectorizer, idf])

ml_model = ml_pipeline.fit(clean_df)
tfidf_df = ml_model.transform(clean_df).cache()

ERROR:root:KeyboardInterrupt while sending command.               (0 + 12) / 19]
Traceback (most recent call last):
  File "/usr/local/lib/python3.10/dist-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/usr/local/lib/python3.10/dist-packages/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
  File "/usr/lib/python3.10/socket.py", line 705, in readinto
    return self._sock.recv_into(b)
KeyboardInterrupt


KeyboardInterrupt: 

In [ ]:
tfidf_df.select("features").show(5, truncate=False)

In [ ]:
clean_df.show(20)

In [ ]:
clean_df.select("tokens").take(5)

In [ ]:
clean_df.coalesce(1).write \
    .option("header", True) \
	.mode("overwrite") \
	.csv("../dataset/spark_preprocessed.csv")